# Two Pointers — Revision Notes

## Core Principle
- Two pointers works when moving one pointer **provably eliminates** possibilities guaranteed to be suboptimal
- Does NOT require a sorted array — requires a logical guarantee that one direction is dead
- Reduces O(n²) to O(n) by never revisiting discarded positions

---

## Problem 1: Two Sum II (Sorted Array)
**Pattern:** Sorted array → sum too big/small tells you which pointer to move
- `sum > target` → right pointer moves left (need smaller)
- `sum < target` → left pointer moves right (need bigger)
- Guarantee: sorted order means moving a pointer monotonically changes the sum

In [2]:
numbers = [2, 7, 11, 15] 
target = 9
left = 0 
right = len(numbers)-1
while left < right : 
    if numbers[left] + numbers[right] > target : 
        right -= 1
    elif numbers[left] + numbers[right] <target : 
        left +=1

    else: 
        print("here")
        print(left, right)
        break 

here
0 1


## Problem 2: Container With Most Water
**Pattern:** NOT sorted — proof is geometric (bottleneck argument)
- Area = `min(height[left], height[right]) * (right - left)`
- Always move the **shorter** side — the shorter side caps the height, and width can only shrink
- **Proof:** Any container keeping the short side with smaller width → min still capped by same height, width smaller → area strictly worse
- Key insight: two pointers works here because the min() constraint creates the elimination guarantee

In [5]:
height = [1,8,6,2,5,4,8,3,7]
# most water needs to be held 
# optimise for min(left, right) * (right - left )
# wait this is not a sorted array we have to think why this even qualifies fro a two pointer appraoch 

# two pointer does not work because its a sorted array
# approach works because moving one pointer gives u a mathematical guarantee that
#  You need a logical guarantee that moving one pointer eliminates possibilities you don't need to check.

most_water = 0 
left = 0 
right = len(height)-1

while left < right : 
    if height[left] < height[right]:
        area = height[left] * (right-left)
        left +=1
    else: 
        area = height[right] * (right-left)
        right -=1
    most_water = max(most_water, area)

print(most_water)

49


## Problem 3: 3Sum
**Pattern:** Fix one element → reduces to Two Sum II on the rest
- **Sort first** — enables two-pointer on the subarray and duplicate skipping
- Compare `nums[left] + nums[right]` against `-nums[i]` (or total sum vs 0)
- **Duplicate skipping at TWO levels:**
  - Outer: `if i > 0 and nums[i] == nums[i-1]: continue` — skip duplicate anchor values
  - Inner: after a match, skip identical left/right values with while loops
- **Traps:**
  - `break` after finding a match → misses other valid pairs for same `i`
  - `continue` without moving a pointer → infinite loop
  - Comparing against `c` instead of `-c`

In [18]:
nums = [-1, 0, 1, 2, -1, -4]
# need to find a+b+c = 0 
# ore a +b = -c 
# not a sorted array 
nums.sort()
for i in range(0, len(nums)):
    c = nums[i]
    left = i+1 
    right = len(nums)-1
    if i>0 and nums[i] == nums[i-1]:
        continue
    while left < right : 
        
        if nums[left] + nums[right] < -c : 
            left+=1
        elif nums[left] + nums[right] > -c : 
            right -=1
        else: 
            print(nums[left], nums[right], c )
            left +=1
            right -=1
            while left < right and nums[left] == nums[left-1]:
                left +=1
            while left < right and nums[right] == nums[right+1]:
                right -=1


-1 2 -1
0 1 -1


## Problem 4: Trapping Rain Water
**Pattern:** Two pointers with running max — process the shorter-max side

**Key formula:** `water[i] = min(left_max, right_max) - height[i]`

**O(n) space version:** Precompute `left_max[]` and `right_max[]` arrays, then sum

**O(1) space two-pointer version:**
- If `height[left] < height[right]` → right side has something taller, so `left_max` is the bottleneck → process left
- If `height[right] <= height[left]` → left side has something taller, so `right_max` is the bottleneck → process right
- Each branch: update its max, add water, move pointer inward
- **Why it works:** When `height[left] < height[right]`, we know `right_max >= height[right] > height[left]`, so `min(left_max, right_max) = left_max` — we can compute water at left without knowing exact right_max

**Traps:**
- `range(n-2, -1)` without step `-1` → empty range, loop doesn't execute
- Mixing up which height to read in each branch (must read own side)
- Moving wrong pointer (left branch moves left, right branch moves right)

In [27]:
height = [0,1,0,2,1,0,1,3,2,1,2,1]
# we maintain a left stack and a right stack for this 
left_max = [0]*len(height)
left_max[0] = height[0]
for i in range(1, len(height)):
    left_max[i] = max(left_max[i-1], height[i])
n = len(height)
right_max = [0]*len(height)
right_max[-1] = height[-1]
for i in range(n-2, -1, -1):
    right_max[i] = max(right_max[i+1], height[i] )

water =0 
for i in range(n ):
    water += min(left_max[i], right_max[i]) - height[i]
water 


6

In [33]:
left = 0 
right = n-1 
left_max = 0 
right_max = 0 
water = 0 
while left < right : 
    if height[left] < height[right]:
       
        left_max = max(left_max, height[left])
        water += left_max - height[left]
        left+=1
        
    else: 
        
        right_max = max(right_max, height[right])
        water += right_max - height[right]
        right -=1
        
water 


6

## Problem 5: Heaters (LC 475)
**Pattern:** Two sorted arrays, one pointer advances monotonically to find closest match

**Why it's two pointers:** Sort both houses and heaters. For each house (pointer 1), advance heater pointer (pointer 2) forward while next heater is closer. Heater pointer never goes back — both sorted means once a heater is past a house, all future houses are even further from previous heaters.

**Key insight:**
- Answer = `max(min distance from each house to its closest heater)`
- Advance heater pointer with a while loop: `while next heater is closer than current, advance`
- After the while loop, current heater is the closest — record distance

**Traps:**
- Forgetting to sort both arrays — two-pointer guarantee breaks without sorting
- Using an `if` instead of `while` to advance heater pointer — closest heater might be 2+ positions forward
- Don't deduplicate with `set()` — unnecessary and removes the guarantee of sorted order if you don't re-sort

**Code pattern:**
```python
houses.sort()
heaters.sort()
j = 0
radius = 0
for i in range(len(houses)):
    while j+1 < len(heaters) and abs(houses[i]-heaters[j]) > abs(houses[i]-heaters[j+1]):
        j += 1
    radius = max(radius, abs(houses[i] - heaters[j]))
return radius
```